# 04. Scoreboard (Image × Regression Cell)

DSC ↔ R² 상관 + §2 가중치 선정(튜닝 UTKFace / held-out SCUT). 이미지 분류 04 노트북 미러(accuracy→R²).

- dataset별 Pearson r / Spearman ρ (합격 단위, plan 20260511-01)
- polluter hold-out, 모델별 r
- 제약 최적화(약신호 상한·핵심 하한) → held-out r — 개선계획 20260530-01 §2

In [ ]:
# ============================================================
# 0. 환경 + 데이터 로드 + merge
# ============================================================
from google.colab import drive; drive.mount('/content/drive')
import os, sys
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
from scipy.optimize import minimize
import json as _json

BASE = '/content/drive/MyDrive/capstone/dsc'
RESULTS_DIR = f'{BASE}/results'
CHARTS_DIR = f'{RESULTS_DIR}/charts_image_regression'
os.makedirs(CHARTS_DIR, exist_ok=True)
if BASE not in sys.path: sys.path.insert(0, BASE)

dsc = pd.read_csv(f'{RESULTS_DIR}/dsc_scores_image_regression.csv')
perf = pd.read_csv(f'{RESULTS_DIR}/model_performance_image_regression.csv')
PERF_Y = 'r2_clipped'   # 회귀 성능 지표 (음수 clip)
merged = perf.merge(dsc[['dataset','polluter','level','score','grade']], on=['dataset','polluter','level'])
merged = merged.rename(columns={'score': 'dsc_score'})
print(f'DSC {len(dsc)}, perf {len(perf)}, merged {len(merged)}')

In [ ]:
# ============================================================
# 1. 산점도 + 등급 박스플롯
# ============================================================
plt.figure(figsize=(10,6))
for m, sub in merged.groupby('model'):
    plt.scatter(sub['dsc_score'], sub[PERF_Y], label=m, alpha=0.6, s=30)
plt.xlabel('DSC Score (image regression cell)'); plt.ylabel('R² (clipped)')
plt.title('DSC ↔ R² 산점도 (image×regression)'); plt.legend(); plt.grid(True, alpha=0.3)
plt.savefig(f'{CHARTS_DIR}/01_scatter.png', dpi=150); plt.show()

In [ ]:
# ============================================================
# 2. 통계 검증 (dataset별 r = 합격 단위) + polluter hold-out + 모델별 r
# ============================================================
THR_R = 0.40
print('[A] dataset별 r — 합격 단위')
dataset_verdict = {}
for ds, sub in merged.groupby('dataset'):
    r, p = pearsonr(sub['dsc_score'], sub[PERF_Y]); rs, ps = spearmanr(sub['dsc_score'], sub[PERF_Y])
    dataset_verdict[ds] = {'r': r, 'rho': rs, 'pass_r': r >= THR_R}
    print(f'  {ds:<14} n={len(sub):>3} r={r:+.4f} p={p:.2e}  ρ={rs:+.4f}  [{"PASS" if r>=THR_R else "FAIL"}]')

print('\n[B] POOLED (보조)')
r_p, _ = pearsonr(merged['dsc_score'], merged[PERF_Y])
print(f'  POOLED n={len(merged)} r={r_p:+.4f}')

print('\n[C] 모델별 r')
model_pos = model_n = 0
for (ds, m), sub in merged.groupby(['dataset','model']):
    if len(sub) < 2: continue
    model_n += 1; rr, _ = pearsonr(sub['dsc_score'], sub[PERF_Y]); model_pos += int(rr > 0)
    print(f'  {ds:>14} | {m:<12s} n={len(sub):>2} r={rr:+.4f}')
print(f'  ▶ 양의 r: {model_pos}/{model_n}')

print('\n[D] Polluter hold-out')
for ds, sub_ds in merged.groupby('dataset'):
    hp_pass = n_pol = 0
    for hp in sorted(sub_ds['polluter'].unique()):
        if hp == 'none': continue
        n_pol += 1; sub = sub_ds[sub_ds.polluter != hp]
        if len(sub) < 3: continue
        rr, _ = pearsonr(sub['dsc_score'], sub[PERF_Y]); hp_pass += int(rr >= THR_R)
    print(f'  {ds}: hold-out {hp_pass}/{n_pol} PASS')

In [ ]:
# ============================================================
# 3. §2 가중치 선정 — 제약 최적화 (튜닝 UTKFace / held-out SCUT)
#    약신호(outlier_ratio·validity·feature_correlation) 상한 0.15,
#    핵심(completeness_image·target_smoothness) 하한 0.10. (개선계획 20260530-01 §2)
# ============================================================
from dsc_framework import DEFAULT_WEIGHTS_IMAGE_REG
METRICS = list(DEFAULT_WEIGHTS_IMAGE_REG.keys())
DEFAULT_W = dict(DEFAULT_WEIGHTS_IMAGE_REG)

merged_full = perf.merge(dsc[['dataset','polluter','level'] + METRICS], on=['dataset','polluter','level'])
tune = merged_full[merged_full.dataset == 'UTKFace']
held = merged_full[merged_full.dataset == 'SCUT_FBP5500']
print(f'튜닝(UTKFace) n={len(tune)}, held-out(SCUT) n={len(held)}')
assert len(tune) >= 4 and len(held) >= 4, '데이터 부족 — 02/03 먼저 실행'

def r_with(X, y, w):
    s = X @ w; return pearsonr(s, y)[0] if s.std() > 1e-12 else 0.0

Xt, yt = tune[METRICS].values, tune[PERF_Y].values
Xh, yh = held[METRICS].values, held[PERF_Y].values
w_def = np.array([DEFAULT_W[m] for m in METRICS])
print(f'\n[Default] 튜닝 r={r_with(Xt,yt,w_def):+.4f}  held-out r={r_with(Xh,yh,w_def):+.4f}')

# 제약: 약신호 상한 0.15, 핵심 하한 0.10, 그 외 [0.02,0.35]
WEAK = {'outlier_ratio','validity','feature_correlation'}
CORE = {'completeness_image','target_smoothness'}
bounds = [(0.02, 0.15) if m in WEAK else (0.02, 0.35) for m in METRICS]
cons = [{'type':'eq','fun': lambda w: w.sum()-1.0}]
for j, m in enumerate(METRICS):
    if m in CORE: cons.append({'type':'ineq','fun': (lambda w, j=j: w[j]-0.10)})
best = None
for seed in range(20):
    w0 = np.random.RandomState(seed).dirichlet(np.ones(len(METRICS)))
    r = minimize(lambda w: -r_with(Xt, yt, w), w0, method='SLSQP', bounds=bounds,
                 constraints=cons, options={'maxiter':600,'ftol':1e-9})
    if best is None or r.fun < best.fun: best = r
w_opt = np.clip(best.x, 0, None); w_opt /= w_opt.sum()
print(f'[제약최적] 튜닝 r={r_with(Xt,yt,w_opt):+.4f}  held-out r={r_with(Xh,yh,w_opt):+.4f}')
print('가중치 top5:', ', '.join(f'{m}={w:.2f}' for m,w in sorted(zip(METRICS,w_opt), key=lambda x:-x[1])[:5]))

out = {'weights': {m: float(w) for m,w in zip(METRICS,w_opt)}, 'tune_dataset':'UTKFace',
       'held_out_dataset':'SCUT_FBP5500', 'perf_metric': PERF_Y,
       'r_default_held_out': float(r_with(Xh,yh,w_def)), 'r_opt_held_out': float(r_with(Xh,yh,w_opt)),
       'note':'image regression 가중치 선정 (제약 최적화, 개선계획 20260530-01 §2).'}
with open(f'{RESULTS_DIR}/tuned_weights_image_regression.json','w',encoding='utf-8') as f:
    _json.dump(out, f, ensure_ascii=False, indent=2)
print('\n저장: tuned_weights_image_regression.json')
print('--- 노트북 04 image regression 완료 ---')